In [ ]:
DATA CLEANING 

 Importing the data

In [2]:
import pandas as pd
import re


Load the data

In [3]:
url = "https://raw.githubusercontent.com/len-chou/TeamBProject/refs/heads/main/GSAF5.csv"

df_shark = pd.read_csv(url)
df_shark.columns = df_shark.columns.str.strip()

df_shark.head()


,Date,Year,Type,Country,State,Location,Activity,Name,Sex,Age,...,Species,Source,pdf,href formula,href,Case Number,Case Number.1,original order,Unnamed: 21,Unnamed: 22
0,18th September,2026,Unprovoked,Australia,Western Australia,Sorrento Beach Perth,Swimming,Greg O'Neil,M,63,...,Great White Shark 6m (20ft),Keith Cowley: Kevin McMurray Trackingsharks.co...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,16th September,2026,Unprovoked,Canada,Quebec,Off the coast of Perce Le Bilbo dive site,Diving,Unknown Male,M,?,...,Great White Shark,Keith Cowley: Kevin McMurray Trackingsharks.co...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,14th September,2026,Unprovoked,Bahamas,Bimini,Bimini Island,Swimming,Unknown Austrian Tourist,F,37,...,Unknown,Kevin McMurray Trackingsharks.com: Keith Cowley,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3,13th September,2026,Unprovoked,Australia,Western Australia,Geraldton,Surfing,Mel Ismail,M,50's,...,Unknown,Simon De Marchi,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
4,7th September,2026,Unprovoked,USA,Hawaii,Honolulu,Surfing,Wants to remain anonymous,M,24,...,Tiger Shark suspected,Keith Cowley,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


In [4]:
1)cleaning first part

SyntaxError: invalid syntax (4207152425.py, line 1)

In [5]:
# 1. Filter dataset to keep only the required columns
selected_columns = [
    'Date', 'Year', 'Type', 'Country', 'State', 
    'Location', 'Activity', 'Fatal Y/N', 'Time'   # 'Activity' added so we can build activity_group later
]
df_clean = df_shark[selected_columns].copy()

# 2. Standardize column names (lowercase, replace spaces/slashes, remove special chars)
df_clean.columns = (
    df_clean.columns
    .str.strip()
    .str.lower()
    .str.replace(' ', '_')
    .str.replace('/', '_')
    .str.replace('?', '')
)

# 3. Clean 'fatal_y_n' column (strip whitespace, uppercase, restrict to 'Y' or 'N')
df_clean['fatal_y_n'] = df_clean['fatal_y_n'].astype(str).str.strip().str.upper()
df_clean['fatal_y_n'] = df_clean['fatal_y_n'].apply(lambda x: x if x in ['Y', 'N'] else pd.NA)

# 4. Clean 'time' column (extract digits and format as HH:MM)
def clean_time(val):
    if pd.isna(val):
        return pd.NA
    
    val_str = str(val).strip().lower()
    digits = re.sub(r'\D', '', val_str)
    
    if len(digits) == 4:
        hours, minutes = digits[:2], digits[2:]
        if 0 <= int(hours) < 24 and 0 <= int(minutes) < 60:
            return f"{hours}:{minutes}"
            
    elif len(digits) == 3:
        hours, minutes = '0' + digits[0], digits[1:]
        if 0 <= int(hours) < 24 and 0 <= int(minutes) < 60:
            return f"{hours}:{minutes}"
            
    return pd.NA

df_clean['time'] = df_clean['time'].apply(clean_time)

# 5. Inspect the cleaned DataFrame
print("--- Data Info ---")
df_clean.info()

print("\n--- First 5 Rows ---")
print(df_clean.head())


--- Data Info ---
<class 'pandas.DataFrame'>
RangeIndex: 7125 entries, 0 to 7124
Data columns (total 9 columns):
 #   Column     Non-Null Count  Dtype
---  ------     --------------  -----
 0   date       7125 non-null   str  
 1   year       7123 non-null   str  
 2   type       7107 non-null   str  
 3   country    7075 non-null   str  
 4   state      6638 non-null   str  
 5   location   6558 non-null   str  
 6   activity   6542 non-null   str  
 7   fatal_y_n  6482 non-null   str  
 8   time       2864 non-null   str  
dtypes: str(9)
memory usage: 1.0 MB

--- First 5 Rows ---
              date  year        type    country              state  \
0   18th September  2026  Unprovoked  Australia  Western Australia   
1   16th September  2026  Unprovoked     Canada             Quebec   
2   14th September  2026  Unprovoked    Bahamas             Bimini   
3  13th September   2026  Unprovoked  Australia  Western Australia   
4    7th September  2026  Unprovoked        USA             H

2) Next we need to  filter to real, recent shark incidents


In [6]:
# Keep only Unprovoked / Provoked / Questionable types, from 1950 onwards.

df_clean['year'] = pd.to_numeric(df_clean['year'], errors='coerce') #here if the value is something else or no value it will be NaN
df_clean['type'] = df_clean['type'].str.strip().str.title()

df_clean = df_clean[
    df_clean['type'].isin(['Unprovoked', 'Provoked', 'Questionable'])
    & (df_clean['year'] >= 1950)
].copy()

print("Shape after filtering:", df_clean.shape)
print("Year range:", df_clean['year'].min(), "to", df_clean['year'].max())
df_clean['type'].value_counts()


Shape after filtering: (4542, 9)
Year range: 1950.0 to 2026.0


type
Unprovoked      4010
Provoked         505
Questionable      27
Name: count, dtype: int64

3) Here we want to simplify the time of the day with 'morning' 'afternoon' 'evening' and 'night' because the dataframe has a lot of mixed values. 

In [7]:
# ---------- Time of day ----------
def hour_to_bucket(t):
    if pd.isna(t):                #pd.isna(t) asks: "is this value empty?" True (yes, empty) or False (no, it has something in it).
        return pd.NA
    hour = int(str(t)[:2])
    if 5 <= hour < 12:
        return 'Morning'
    if 12 <= hour < 17:
        return 'Afternoon'
    if 17 <= hour < 21:
        return 'Evening'
    else:
        return 'Night'

def word_to_bucket(raw):
    if pd.isna(raw):
        return pd.NA
    text = str(raw).lower()
    if 'afternoon' in text or 'midday' in text or 'noon' in text or 'p.m' in text:
        return 'Afternoon'
    if 'morning' in text or 'dawn' in text or 'a.m' in text:
        return 'Morning'
    if 'evening' in text or 'dusk' in text or 'sunset' in text:
        return 'Evening'
    if 'night' in text:
        return 'Night'
    return pd.NA

df_clean['time_bucket'] = df_clean['time'].apply(hour_to_bucket)
original_time = df_shark.loc[df_clean.index, 'Time']
df_clean['time_bucket'] = df_clean['time_bucket'].fillna(original_time.apply(word_to_bucket))


Rows with a time of day: 2848 of 4542
Rows with a month: 4349 of 4542

activity_group
Surfing               1227
Swimming/wading       1102
Diving/snorkeling      499
Spearfishing           463
Fishing                423
Unknown                263
Other                  220
Bodyboarding           194
Kayak/paddle            88
Foil/kite/windsurf      63
Name: count, dtype: int64


In [ ]:
# ---------- Month ----------
months = {'jan': 1, 'feb': 2, 'mar': 3, 'apr': 4, 'may': 5, 'jun': 6,
          'jul': 7, 'aug': 8, 'sep': 9, 'oct': 10, 'nov': 11, 'dec': 12}

def get_month(text):
    if pd.isna(text):
        return pd.NA
    text = str(text).lower()
    found = re.search(r'(jan|feb|mar|apr|may|jun|jul|aug|sep|oct|nov|dec)', text)
    if found:
        return months[found.group(1)]
    dotted = re.match(r'\s*\d{4}\.(\d{2})\.', text)   # dates like 2017.06.05
    if dotted and 1 <= int(dotted.group(1)) <= 12:
        return int(dotted.group(1))
    return pd.NA

df_clean['month'] = df_clean['date'].apply(get_month)


In [ ]:
Here we are taking a messy column called 'activity' and putting each value into a smaller number of generalized groups.

In [8]:
# ---------- Activity groups ----------
def activity_group(a):
    if pd.isna(a):
        return 'Unknown'
    t = str(a).lower().strip()
    if t in ('', '?', 'not stated', 'unknown', 'unconfirmed'):
        return 'Unknown'
    if 'spear' in t:
        return 'Spearfishing'
    if 'fish' in t:
        return 'Fishing'
    if any(k in t for k in ['windsurf', 'kite', 'foil', 'wing']):
        return 'Foil/kite/windsurf'
    if ('body' in t and ('board' in t or 'surf' in t)) or 'boogie' in t:
        return 'Bodyboarding'
    if 'surf ski' in t or 'surfski' in t:
        return 'Kayak/paddle'
    if 'surf' in t:
        return 'Surfing'
    if any(k in t for k in ['scuba', 'diving', 'dive', 'snorkel', 'skin']):
        return 'Diving/snorkeling'
    if any(k in t for k in ['kayak', 'canoe', 'paddl', 'sup ', 'rowing', 'raft']):
        return 'Kayak/paddle'
    if any(k in t for k in ['swim', 'bath', 'wad', 'stand', 'float', 'tread', 'play', 'splash', 'walk', 'cool']):
        return 'Swimming/wading'
    return 'Other'

df_clean['activity_group'] = df_clean['activity'].apply(activity_group)

# ---------- Check ----------
print("Rows with a time of day:", df_clean['time_bucket'].notna().sum(), "of", len(df_clean))
print("Rows with a month:", df_clean['month'].notna().sum(), "of", len(df_clean))
print()
print(df_clean['activity_group'].value_counts())


Rows with a time of day: 2848 of 4542
Rows with a month: 4349 of 4542

activity_group
Surfing               1227
Swimming/wading       1102
Diving/snorkeling      499
Spearfishing           463
Fishing                423
Unknown                263
Other                  220
Bodyboarding           194
Kayak/paddle            88
Foil/kite/windsurf      63
Name: count, dtype: int64
